In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage.io
import os 
import tqdm
import glob
import tensorflow 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.data.experimental import AUTOTUNE
from tensorflow.keras import Sequential, Input, Model
from tensorflow.keras.layers import RandomRotation, RandomZoom
from tensorflow.keras.layers.experimental.preprocessing import Rescaling
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras import applications
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from keras import layers

from tqdm import tqdm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from skimage.io import imread, imshow
from skimage.transform import resize

from tensorflow.keras.models import Sequential
from tensorflow.keras.metrics import Precision, AUC,Recall
from tensorflow.keras.layers import InputLayer, BatchNormalization, Dropout, Flatten, Dense, Activation, MaxPool2D, Conv2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications.densenet import DenseNet169
import copy
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
import cv2
import keras
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img, img_to_array
import matplotlib
import matplotlib.pylab as plt
import seaborn as sns
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix
from keras.applications.resnet import ResNet50,preprocess_input
from keras.applications.resnet import ResNet50,preprocess_input
from keras.applications import ResNet50V2
from keras.applications import ResNet50V2
from tensorflow.keras.utils import image_dataset_from_directory
import tensorflow_addons as tfa
from tensorflow_addons.optimizers import RectifiedAdam
from tensorflow_addons.optimizers import AdamW
from PIL import Image

labels = pd.read_csv('AlzheimersFLAIR/patient_labels.csv')
labels

In [ ]:
#tf.config.run_functions_eagerly(False)

class CustomDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, directory, batch_size):
        self.volume_tensor, self.label_list = self.load_and_preprocess_images(directory)
        self.batch_size = batch_size
        #self.weighted = weighted
    
    def __len__(self):
        return int(np.ceil(len(self.volume_tensor) / self.batch_size))
    
    def __getitem__(self, index):
        batch_indices = []
        #if self.weighted:
        batch_indices = self.get_weighted_random_sample()
        
        #else:
        batch_indices = np.random.choice(np.arange(len(self.label_list)), size = batch_size, replace = False)
        #batch_x = tf.gather(self.volume_tensor, batch_indices)
        batch_x = np.stack([self.volume_tensor[i] for i in batch_indices])
        
        batch_y = [self.label_list[i] for i in batch_indices]
        
        out_y = []
        for i in batch_y:
            label = [0, 0, 0, 0]
            label[i] += 1
            out_y.append(np.array(label))
        
        return batch_x, np.array(out_y)
    
    def load_and_preprocess_images(self, directory):
        # load images
        label_list = []
        image_list = []
        for label in os.listdir(directory):
            #print(label)
            if (label != ".DS_Store"):
                label_path = os.path.join(directory, label)
                
                for img in os.listdir(label_path):
                    if (img.endswith('.jpg')):
                        img_path = os.path.join(label_path, img)
                        
                        label_list.append(int(label))
                        image_list.append(np.expand_dims(np.array(Image.open(img_path)), axis = -1))
        image_tensor = tf.image.grayscale_to_rgb(tf.convert_to_tensor(image_list))
        #rgb_list = [np.stack((img,) * 3, axis=-1) for img in image_list]
        
        data_augmentor = Sequential([
            layers.Resizing(224, 224),
            layers.Rescaling(1./255),
            layers.RandomRotation(0.1, fill_mode = 'constant'),
            #layers.RandomZoom((.5, .5)),
            # randomzoom without using 'constant' interpolates with corners of the brain
            # using 'constant' makes randomzoom not actually zoom
        ])

#        augmented_list = []
#        for i in rgb_list:
#            augmented_list.append(data_augmentor(i))

        augmented_list = []
        for i in image_tensor:
            augmented_list.append(data_augmentor(i).numpy())
        print(type(augmented_list[0]))
        tensors = np.stack(augmented_list)
        #print(np.shape(tensors)

#         plt.figure()
#         plt.imshow(augmented_list[0])
#         plt.show()

        return tensors, label_list
        
    
    def get_weighted_random_sample(self):
        class_frequencies = np.bincount(self.label_list)

        # Calculate the inverse of class frequencies as probabilities
        class_probabilities = 1.0 / class_frequencies
        #print(class_probabilities)
        # Normalize the probabilities to sum to 1
        class_probabilities /= np.sum(class_probabilities)
        #print(self.label_list)
        index_probabilities = [class_probabilities[i] for i in self.label_list]
        
        index_probabilities /= np.sum(index_probabilities)
        
        # Randomly choose sample indices with replacement using class probabilities
        random_indices = np.random.choice(np.arange(len(self.label_list)), size=self.batch_size, 
                                          replace=False, p=index_probabilities)
        return random_indices

In [ ]:
batch_size = 16
train_dir = 'Splitted2D/train'
val_dir = 'Splitted2D/val'
test_dir = 'Splitted2D/test'

train_generator = CustomDataGenerator(directory = train_dir, batch_size = batch_size)
val_generator = CustomDataGenerator(directory = val_dir, batch_size = batch_size)
test_generator = CustomDataGenerator(directory = test_dir, batch_size = batch_size)

In [ ]:
input_shape = (224,224, 3)

#Create an instance of the VGG16 model
resnet50 = tf.keras.applications.ResNet50V2(include_top=False, input_shape=input_shape,
                   weights='imagenet')

# Freeze the layers of the VGG16 model
for layer in resnet50.layers:
    layer.trainable = False
    
# Create a new model with additional layers
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()

prediction_layer = keras.layers.Dense(4,activation='softmax')

model = tf.keras.Sequential([resnet50, global_average_layer,
  keras.layers.BatchNormalization(),  
    keras.layers.Dropout(0.3),
    keras.layers.Dense(2048, activation='relu'),
    keras.layers.Dropout(0.25),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.15),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.1),
    prediction_layer
])

model.compile(optimizer=AdamW(learning_rate = 0.0001, weight_decay = 0.01), loss='categorical_crossentropy', 
                        metrics=['accuracy',tf.keras.metrics.AUC(),
                        tf.keras.metrics.Precision(),
                        tf.keras.metrics.Recall(),])

In [ ]:
resnet50.summary()

In [ ]:

history=model.fit(train_generator,
                        validation_data=val_generator,
                        steps_per_epoch=len(train_generator),
                        epochs = 100,
                        verbose = 1)

In [ ]:
result = model.evaluate(train_generator)
train_loss = result[0]
train_accuracy = result[1]
train_AUC = result[2]
train_pre = result[3]
train_rec = result[4]
print(f'Train Loss = {train_loss}')
print(f'Train Accuracy = {train_accuracy}')
print(f'Train AUC = {train_AUC}')
print(f'Train Precision = {train_pre}')
print(f'Train Recall = {train_rec}')

result = model.evaluate(test_generator)
test_loss = result[0]
test_accuracy = result[1]
test_AUC = result[2]
test_pre = result[3]
test_rec = result[4]
print(f'Test Loss = {test_loss}')
print(f'Test Accuracy = {test_accuracy}')
print(f'Test AUC = {test_AUC}')
print(f'Test Precision = {test_pre}')
print(f'Test Recall = {test_rec}')